# 02 - Hiperparámetros, Interpretabilidad y Partial Dependence Plots (PDP)

## Curso de Machine Learning: Modelos de Regresión

En este cuaderno estudiaremos la calibración sistemática de un `RandomForestRegressor`, contrastaremos la importancia de características basada en reducción de impureza (**MDI**) frente a la importancia por **Permutación**, y utilizaremos los **Gráficos de Dependencia Parcial (PDP)** para interpretar de forma continua el impacto de cada variable sobre la predicción continua.

---

### Contenidos
1. Hiperparámetros Clave en Regresión (`n_estimators`, `max_features`, `min_samples_leaf`)
2. Optimización con `GridSearchCV`
3. Interpretabilidad Global: MDI vs. Permutation Importance
4. Gráficos de Dependencia Parcial (PDP) con `PartialDependenceDisplay`
5. Visualización 1D del Ajuste Continuo Suavizado
6. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import r2_score, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Carga y Preparación de Datos Auto MPG

In [ ]:
df = pd.read_csv("../../data/auto_mpg.csv")
features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
X = df[features]
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


## 2. Búsqueda de Hiperparámetros con `GridSearchCV`

En regresión con Random Forest, los hiperparámetros con mayor impacto son:
- `max_features`: Controla la descorrelación de los árboles (`1.0`, `'sqrt'`, `0.5`).
- `min_samples_leaf`: Evita hojas excesivamente pequeñas con predicciones inestables ($1, 2, 4$).
- `max_depth`: Limita la profundidad de los árboles individuales.


In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_features': ['sqrt', 1.0, 0.6],
    'min_samples_leaf': [1, 2, 4]
}

grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)

print(f"Mejores Hiperparametros: {grid_rf.best_params_}")
print(f"Mejor R2 CV: {grid_rf.best_score_:.4f}")

best_rf = grid_rf.best_estimator_
y_pred = best_rf.predict(X_test)
print(f"R2 en Prueba: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE en Prueba: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} mpg")


## 3. Interpretabilidad: MDI vs. Permutation Importance

En regresión, **MDI (Mean Decrease in Impurity)** suma la reducción de varianza (MSE) en cada división. Sin embargo, puede sobrestimar variables continuas con alta cardinalidad. Por ello, contrastamos con **Permutation Importance** sobre el conjunto de prueba independiente.


In [ ]:
# 1. MDI (feature_importances_)
importancia_mdi = pd.Series(best_rf.feature_importances_, index=features).sort_values(ascending=False)

# 2. Permutation Importance en Test
perm = permutation_importance(best_rf, X_test, y_test, n_repeats=15, random_state=42, scoring='r2')
importancia_perm = pd.Series(perm.importances_mean, index=features).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

importancia_mdi.plot(kind='barh', ax=axes[0], color='#2b5c8f')
axes[0].set_title('Importancia MDI (Reduccion Varianza MSE)', fontweight='bold')
axes[0].invert_yaxis()

importancia_perm.plot(kind='barh', ax=axes[1], color='#2ca02c')
axes[1].set_title('Permutation Importance en Test (Caida en R²)', fontweight='bold')
axes[1].invert_yaxis()

plt.suptitle('Comparativa de Importancia de Variables en Random Forest Regressor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Gráficos de Dependencia Parcial (PDP)

Un **Partial Dependence Plot (PDP)** muestra la relación marginal entre una o dos características y la variable objetivo predicha, manteniendo el resto de las variables constantes o marginalizadas:

$$f_{pdp}(x_s) = \mathbb{E}_{X_c} [\hat{f}(x_s, X_c)] \approx \frac{1}{N} \sum_{i=1}^N \hat{f}(x_s, \mathbf{x}_{i, c})$$


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
PartialDependenceDisplay.from_estimator(
    best_rf, X_train,
    features=['weight', 'displacement', 'model_year'],
    ax=ax,
    line_kw={'color': '#d95f02', 'linewidth': 2.5}
)
plt.suptitle('Graficos de Dependencia Parcial (PDP): Efecto Marginal sobre el Consumo (mpg)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Visualización 1D: Árbol vs. Random Forest

Observamos cómo Random Forest transforma la función escalonada y rígida de un árbol en una curva suave continua:


In [ ]:
# Ajuste unidimensional usando solo 'weight' para predecir 'mpg'
X_1d = df[['weight']].values
y_1d = df['mpg'].values

x_grid = np.linspace(X_1d.min(), X_1d.max(), 500).reshape(-1, 1)

tree_1d = DecisionTreeRegressor(max_depth=4, random_state=42).fit(X_1d, y_1d)
rf_1d = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_1d, y_1d)

plt.figure(figsize=(11, 5))
plt.scatter(X_1d, y_1d, color='gray', alpha=0.5, s=30, label='Muestras Reales')
plt.plot(x_grid, tree_1d.predict(x_grid), color='#d62728', lw=2, label='Arbol Unico (Escalones rigidos)')
plt.plot(x_grid, rf_1d.predict(x_grid), color='#2b5c8f', lw=2.5, label='Random Forest (Curva Suavizada)')

plt.xlabel('Peso del Vehiculo (weight en lbs)', fontweight='bold')
plt.ylabel('Millas por Galon (mpg)', fontweight='bold')
plt.title('Comparativa de Curvas de Ajuste 1D: Eliminacion de Discontinuidades Abruptas', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 6. Conclusiones

1. **Hiperparámetros**: `max_features` y `min_samples_leaf` son las palancas principales para regular la suavidad del modelo.
2. **PDP**: Permite responder preguntas de negocio como *"¿cuánto se reduce el rendimiento al aumentar 500 lbs de peso?"*.
3. **Consistencia**: Random Forest supera al árbol individual eliminando saltos bruscos y aumentando la precisión en datos no vistos.
